<a href="https://colab.research.google.com/github/Karansn-dev/Cancer_Risk-Prediction-ML-/blob/main/Cancer_Risk_Prediction_(ML).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

In [2]:
from google.colab import files
uploaded = files.upload()

Saving cancer-risk-factors.csv to cancer-risk-factors.csv


In [3]:
#preparing the data

df = pd.read_csv('/content/cancer-risk-factors.csv')

In [4]:
X = df.drop(columns = ['Risk_Level' ,'Patient_ID' , 'Cancer_Type' ])
#Encode the target variable

le = LabelEncoder()
y = le.fit_transform(df['Risk_Level'])

In [5]:
print(le.classes_)

['High' 'Low' 'Medium']


In [6]:
y

array([2, 2, 2, ..., 1, 2, 2], shape=(2000,))

In [7]:
#split into trainig/test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
#feature Scaling

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
model = RandomForestClassifier(random_state=42,n_estimators=100)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [10]:
y_pred = model.predict(X_test)

print("Classification Report ")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))


Classification Report 
              precision    recall  f1-score   support

        High       1.00      0.95      0.97        20
         Low       1.00      1.00      1.00        65
      Medium       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       1.00      0.98      0.99       400
weighted avg       1.00      1.00      1.00       400

Confusion Matrix
[[ 19   0   1]
 [  0  65   0]
 [  0   0 315]]


####The model correctly classified all Low- and Medium-risk records. It missed one High-risk record by predicting it as Medium.

#Logistic Regression Model

In [11]:
# Create the model
model_lr = LogisticRegression(random_state=42)

# Train the model
model_lr.fit(X_train, y_train)

# Make predictions
y_pred_lr = model_lr.predict(X_test)

# Evaluate the model
print("Classification Report")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_lr))


Classification Report
              precision    recall  f1-score   support

        High       1.00      0.80      0.89        20
         Low       0.97      0.95      0.96        65
      Medium       0.98      0.99      0.99       315

    accuracy                           0.98       400
   macro avg       0.98      0.92      0.95       400
weighted avg       0.98      0.98      0.98       400

Confusion Matrix
[[ 16   0   4]
 [  0  62   3]
 [  0   2 313]]


In [12]:
X = df.drop(columns = ['Risk_Level' ,'Patient_ID' , 'Cancer_Type', 'Overall_Risk_Score' ])
#Encode the target variable

le_new = LabelEncoder()
y_new = le_new.fit_transform(df['Risk_Level'])

In [13]:

# Split into training and testing sets
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Feature Scaling
scaler_new = StandardScaler()

X_train_new = scaler_new.fit_transform(X_train_new)
X_test_new = scaler_new.transform(X_test_new)

# Create and train the Random Forest model
model_new = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

model_new.fit(X_train_new, y_train_new)

# Make predictions
y_pred_new = model_new.predict(X_test_new)

# Evaluate the model
print("Classification Report")
print(classification_report(
    y_test_new,
    y_pred_new,
    target_names=le.classes_
))

print("Confusion Matrix")
print(confusion_matrix(y_test_new, y_pred_new))


Classification Report
              precision    recall  f1-score   support

        High       1.00      0.05      0.10        20
         Low       0.85      0.34      0.48        65
      Medium       0.83      0.99      0.90       315

    accuracy                           0.83       400
   macro avg       0.89      0.46      0.49       400
weighted avg       0.84      0.83      0.80       400

Confusion Matrix
[[  1   0  19]
 [  0  22  43]
 [  0   4 311]]


In [14]:
from imblearn.over_sampling import SMOTE

In [15]:
smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(
    X_train_new,
    y_train_new
)

print("Class distribution after SMOTE:")
print(pd.Series(y_train_res).value_counts())

Class distribution after SMOTE:
2    1259
1    1259
0    1259
Name: count, dtype: int64


In [16]:
rf_model = RandomForestClassifier (random_state=42)
rf_model.fit(X_train_res, y_train_res)

RandomForestClassifier(random_state=42)

In [17]:
y_pred_2 = rf_model.predict(X_test_new)
print("Classification Report")
print(classification_report(y_test_new, y_pred_2))

Classification Report
              precision    recall  f1-score   support

           0       0.62      0.25      0.36        20
           1       0.72      0.63      0.67        65
           2       0.88      0.94      0.91       315

    accuracy                           0.85       400
   macro avg       0.74      0.61      0.65       400
weighted avg       0.84      0.85      0.84       400



#OPTA TUNING


In [19]:
%pip install -q optuna

In [20]:
# Optuna tuning
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

import numpy as np
import joblib  # optional, to save model/study

In [21]:
X_train_res.shape

(3777, 17)

In [22]:
from sklearn.metrics import f1_score, make_scorer

# Define macro F1 scorer explicitly for multiclass
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

scorer = make_scorer(macro_f1)

In [26]:
X_train_np = np.asarray(X_train_res)
y_train_np = np.asarray(y_train_res)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestClassifier(**params)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train_np, y_train_np):
        X_t, X_v = X_train_np[train_idx], X_train_np[val_idx]
        y_t, y_v = y_train_np[train_idx], y_train_np[val_idx]

        model.fit(X_t, y_t)
        y_pred = model.predict(X_v)

        score = f1_score(y_v, y_pred, average='macro')
        scores.append(score)

    return np.mean(scores)

In [27]:
# --- Create study ---
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, study_name='rf_macro_f1')


[I 2026-11-19 17:15:02,706] A new study created in memory with name: rf_macro_f1


In [28]:
# --- Run optimization ---
n_trials = 50  # change to 100+ if you want more thorough search
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("Best trial:")
print("  Value (macro F1):", study.best_value)
print("  Params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-11-19 17:15:14,618] Trial 0 finished with value: 0.944607455324415 and parameters: {'n_estimators': 218, 'max_depth': 29, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 0 with value: 0.944607455324415.
[I 2026-11-19 17:15:16,907] Trial 1 finished with value: 0.9488367157392911 and parameters: {'n_estimators': 59, 'max_depth': 30, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 1 with value: 0.9488367157392911.
[I 2026-11-19 17:15:22,774] Trial 2 finished with value: 0.9185777345721403 and parameters: {'n_estimators': 325, 'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 1 with value: 0.9488367157392911.
[I 2026-11-19 17:15:26,749] Trial 3 finished with value: 0.9189867064795951 and parameters: {'n_estimators': 324, 'max_depth'

In [29]:
# Train final model with best params and full training data, then evaluate on test
best_params = {
    'n_estimators': 347,
    'max_depth': 19,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'bootstrap': False,
    'criterion': 'gini',
    'random_state': 42,
    'n_jobs': -1
}


final_rf = RandomForestClassifier(**best_params)
final_rf.fit(X_train_res, y_train_res)         # resampled training set
y_pred_test = final_rf.predict(X_test)        # untouched test set

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_test))
print(confusion_matrix(y_test, y_pred_test))


ValueError: X has 18 features, but RandomForestClassifier is expecting 17 features as input.

In [34]:
print("X_train_res shape:", X_train_res.shape)
print("X_test shape:", X_test.shape)

# If they are DataFrames, find the exact extra column:
if hasattr(X_train_res, 'columns') and hasattr(X_test, 'columns'):
    print("Extra in X_test:", set(X_test.columns) - set(X_train_res.columns))
    print("Missing from X_test:", set(X_train_res.columns) - set(X_test.columns))

X_train_res shape: (3777, 17)
X_test shape: (400, 18)


In [32]:
for name, val in list(globals().items()):
    if (name.startswith("X_") or name.startswith("x_")) and hasattr(val, "shape"):
        print(f"{name}: {val.shape} ({type(val).__name__})")

X_train: (1600, 18) (ndarray)
X_test: (400, 18) (ndarray)
X_train_new: (1600, 17) (ndarray)
X_test_new: (400, 17) (ndarray)
X_train_res: (3777, 17) (ndarray)
X_train_np: (3777, 17) (ndarray)


In [33]:
best_params = {
    'n_estimators': 347,
    'max_depth': 19,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'bootstrap': False,
    'criterion': 'gini',
    'random_state': 42,
    'n_jobs': -1
}

final_rf = RandomForestClassifier(**best_params)
final_rf.fit(X_train_res, y_train_res)          # resampled training set (17 features)
y_pred_test = final_rf.predict(X_test_new)      # matching test set (17 features)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_test))
print(confusion_matrix(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.67      0.20      0.31        20
           1       0.76      0.58      0.66        65
           2       0.88      0.96      0.91       315

    accuracy                           0.86       400
   macro avg       0.77      0.58      0.63       400
weighted avg       0.85      0.86      0.84       400

[[  4   0  16]
 [  0  38  27]
 [  2  12 301]]


In [35]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, make_scorer, recall_score
import optuna
import numpy as np

# scorer (macro f1)
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

macro_f1_scorer = make_scorer(macro_f1)

def objective(trial):
    # RF params to try
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 6, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
        'max_features': trial.suggest_categorical('max_features', ['sqrt','log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini','entropy']),
        'random_state': 42,
        'n_jobs': 1   # avoid nested parallelism inside cross_val_score
    }

    sm = SMOTE(random_state=42)
    rf = RandomForestClassifier(**params)
    pipe = ImbPipeline([('smote', sm), ('rf', rf)])

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=macro_f1_scorer, n_jobs=1)
    return float(np.mean(scores))

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)
print(study.best_value, study.best_params)



[I 2026-11-19 17:31:56,549] A new study created in memory with name: no-name-411cd4c9-8ef2-4641-bf2b-9716bb3bc981
[I 2026-11-19 17:32:02,223] Trial 0 finished with value: 0.9912410751814288 and parameters: {'n_estimators': 212, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9912410751814288.
[I 2026-11-19 17:32:04,026] Trial 1 finished with value: 0.9956950453201157 and parameters: {'n_estimators': 106, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 1 with value: 0.9956950453201157.
[I 2026-11-19 17:32:08,651] Trial 2 finished with value: 0.9956950453201157 and parameters: {'n_estimators': 284, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 1 with value: 0.9956950453201157.
[I 2026-

0.9978473651894392 {'n_estimators': 282, 'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'gini'}


In [36]:
# Suppose best_params from study:
best = study.best_params
rf = RandomForestClassifier(**best, random_state=42, n_jobs=-1)

pipe_final = ImbPipeline([('smote', SMOTE(random_state=42)), ('rf', rf)])
pipe_final.fit(X_train, y_train)   # fit on original X_train (pipeline will SMOTE inside)

y_test_pred = pipe_final.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.98      1.00      0.99        65
           2       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       0.99      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

[[ 20   0   0]
 [  0  65   0]
 [  0   1 314]]


## **XGBoost**

In [37]:
# baseline XGBoost (SMOTE inside pipeline - no leakage)

from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline

le = LabelEncoder()
le.fit(y_train)

y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

# Build Pipeline
xgb = XGBClassifier(use_label_encoder=False, eval_metrics='mlogloss', random_state=42, n_jobs=-1)
pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', xgb)
])

pipe.fit(X_train, y_train_enc)

y_pred_enc = pipe.predict(X_test)

y_pred = le.inverse_transform(y_pred_enc)
y_test_orig = le.inverse_transform(y_test_enc)

print("Baseline XGB Classifier")
print(classification_report(y_test_orig, y_pred))
print("Confusion Matrix")
print(confusion_matrix(y_test_orig, y_pred))

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [17:37:13] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Baseline XGB Classifier
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.97      1.00      0.98        65
           2       1.00      0.99      1.00       315

    accuracy                           0.99       400
   macro avg       0.99      1.00      0.99       400
weighted avg       1.00      0.99      1.00       400

Confusion Matrix
[[ 20   0   0]
 [  0  65   0]
 [  0   2 313]]


In [41]:
# --- Imports ---
import optuna
from optuna.samplers import TPESampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, classification_report, confusion_matrix
import numpy as np

# --- Settings ---
TARGET_LABEL = 0       # 0 = minority/High class (matches le.classes_ = [0, 1, 2])
N_TRIALS = 40
CV_FOLDS = 3

# Use X_train_new / X_test_new (17 features) if X_train has 18 features
X_train_arr = np.asarray(X_train_new)
X_test_arr = np.asarray(X_test_new)
y_train_arr = np.asarray(y_train)
y_test_arr = np.asarray(y_test)

classes = np.unique(y_train_arr)
target_index = int(np.flatnonzero(classes == TARGET_LABEL)[0])

# --- Objective: maximize recall for TARGET_LABEL ---
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 1e-2, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42,
        'eval_metric': 'mlogloss'
    }

    xgb = XGBClassifier(**params, n_jobs=1)
    pipe = ImbPipeline([('smote', SMOTE(random_state=42)), ('xgb', xgb)])

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    recalls = []

    for train_idx, val_idx in cv.split(X_train_arr, y_train_arr):
        X_t, X_v = X_train_arr[train_idx], X_train_arr[val_idx]
        y_t, y_v = y_train_arr[train_idx], y_train_arr[val_idx]

        pipe.fit(X_t, y_t)
        y_pred = pipe.predict(X_v)

        recs = recall_score(
            y_v, y_pred,
            labels=classes,
            average=None,
            zero_division=0
        )
        recalls.append(recs[target_index])

    return float(np.mean(recalls))

# --- Run Optuna study ---
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, study_name='xgb_class0_recall')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nBest CV mean Class-0 recall:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# --- Train final pipeline with best params and evaluate on test set ---
best_params = study.best_params.copy()
best_params.update({'eval_metric': 'mlogloss', 'random_state': 42})
final_xgb = XGBClassifier(**best_params, n_jobs=-1)

final_pipe = ImbPipeline([('smote', SMOTE(random_state=42)), ('xgb', final_xgb)])
final_pipe.fit(X_train_arr, y_train_arr)

y_test_pred = final_pipe.predict(X_test_arr)

print("\nFINAL Test Classification Report:")
print(classification_report(y_test_arr, y_test_pred, labels=classes))
print("Confusion Matrix (rows=true, cols=predicted):")
print(confusion_matrix(y_test_arr, y_test_pred, labels=classes))

[I 2026-11-19 17:41:03,245] A new study created in memory with name: xgb_class0_recall


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-11-19 17:41:05,258] Trial 0 finished with value: 0.4519400352733686 and parameters: {'n_estimators': 181, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.4936111842654619, 'gamma': 0.7799726016810132, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 0.4519400352733686.
[I 2026-11-19 17:41:16,028] Trial 1 finished with value: 0.4757495590828924 and parameters: {'n_estimators': 260, 'max_depth': 10, 'learning_rate': 0.010725209743171997, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.899465584480253, 'gamma': 1.0616955533913808, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 1 with value: 0.4757495590828924.
[I 2026-11-19 17:41:18,563] Trial 2 finished with value: 0.46428571428571425 and parameters: {'n_estimators': 156, 'max_depth': 8, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.7671

In [51]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 1. Encode target labels (if not already numeric)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# 2. Compute class weights inversely proportional to class frequencies
classes, counts = np.unique(y_train_enc, return_counts=True)
class_weights = {cls: max(counts) / count for cls, count in zip(classes, counts)}
print("Class Weights:", class_weights)

# 3. Initialize XGBoost with weights
xgb_weighted = XGBClassifier(
    objective='multi:softmax',
    num_class=len(classes),
    eval_metric='mlogloss',
    random_state=42,
    n_estimators=105,
    max_depth=3,
    learning_rate=0.014843806162322944,
    subsample=0.7730982444721965,
    colsample_bytree=0.8509237341976973,
    gamma=4.134336554829186,
    reg_alpha=4.453508368364819,
    reg_lambda=2.616923339470738,
    n_jobs=-1
)

# 4. Fit model with per-sample weights (swap to X_train_new if using 17 features)
sample_weights = np.array([class_weights[y] for y in y_train_enc])
xgb_weighted.fit(X_train, y_train_enc, sample_weight=sample_weights)

# 5. Evaluate (swap to X_test_new if using 17 features)
y_pred_enc = xgb_weighted.predict(X_test)
print("Class-weighted XGBoost - Classification Report")
print(classification_report(
    y_test_enc,
    y_pred_enc,
    target_names=[str(c) for c in le.classes_]
))
print("Confusion Matrix")
print(confusion_matrix(y_test_enc, y_pred_enc))

Class Weights: {np.int64(0): np.float64(15.353658536585366), np.int64(1): np.float64(4.861003861003861), np.int64(2): np.float64(1.0)}
Class-weighted XGBoost - Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.98      0.98      0.98        65
           2       1.00      1.00      1.00       315

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400

Confusion Matrix
[[ 20   0   0]
 [  0  64   1]
 [  0   1 314]]


In [43]:
print(classification_report(y_test_enc, y_pred_enc, target_names=[str(c) for c in le.classes_]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.98      0.98      0.98        65
           2       1.00      1.00      1.00       315

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400



In [44]:
import pandas as pd

# Top 10 most important features in xgb_weighted
importances = pd.Series(xgb_weighted.feature_importances_).sort_values(ascending=False)
print("Top features by importance (index):")
print(importances.head(10))

Top features by importance (index):
15    0.582077
10    0.164953
3     0.045210
7     0.033924
2     0.030114
6     0.027235
11    0.024437
13    0.022887
4     0.016884
8     0.015847
dtype: float32


In [45]:
misclassified_idx = np.where(y_test_enc != y_pred_enc)[0]
print("Misclassified indices:", misclassified_idx)
for idx in misclassified_idx:
    print(f"Row {idx}: true={y_test_enc[idx]}, pred={y_pred_enc[idx]}")

Misclassified indices: [198 328]
Row 198: true=2, pred=1
Row 328: true=1, pred=2


In [46]:
import joblib

artifact = {
    "model": xgb_weighted,
    "label_encoder": le,
    "class_weights": class_weights,
}
joblib.dump(artifact, "xgb_weighted_model.joblib")

['xgb_weighted_model.joblib']

In [47]:
# If you have a list of feature names (17 items for X_train_new):
# print("Feature 15 name:", feature_names[15])

print("Row 198 features:", X_test_new[198])
print("Row 328 features:", X_test_new[328])

Row 198 features: [-0.78009296 -0.97652545 -0.64579754 -1.2288373   0.99087203 -0.48727028
  1.21391088 -0.4884331  -0.32118293  1.0251683  -1.67360922 -1.23125659
 -0.17396553 -0.4843221   0.35203998 -0.34488102  1.28947119]
Row 328 features: [-0.40074527 -0.97652545 -1.54560253  0.60353892  0.33696597 -0.48727028
 -0.05928309 -0.4884331  -0.65144815 -0.99819019 -1.36005246 -0.29449456
 -0.17396553 -0.4843221  -1.28250432 -2.15084048  0.97424523]


In [48]:
import joblib

loaded = joblib.load("xgb_weighted_model.joblib")
preds = loaded["model"].predict(X_test_new)
print("Matches y_pred_enc:", (preds == y_pred_enc).all())

ValueError: Feature shape mismatch, expected: 18, got 17

In [49]:
import joblib

loaded = joblib.load("xgb_weighted_model.joblib")
preds = loaded["model"].predict(X_test)
print("Matches y_pred_enc:", (preds == y_pred_enc).all())

Matches y_pred_enc: True


In [53]:
# Optuna tuning for class-weighted XGBoost (optimize macro-F1)
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Convert feature matrices to NumPy arrays for consistent bracket indexing
X_train_arr = np.asarray(X_train)
X_test_arr = np.asarray(X_test)

# ----------------------
# Prepare encoded labels
# ----------------------
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
n_classes = len(le.classes_)

# ----------------------
# Objective: maximize macro-F1
# ----------------------
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'eval_metric': 'mlogloss',
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train_arr, y_train_enc):
        X_t, X_v = X_train_arr[train_idx], X_train_arr[val_idx]
        y_t, y_v = y_train_enc[train_idx], y_train_enc[val_idx]

        # Compute class weights from the fold's training data
        classes, counts = np.unique(y_t, return_counts=True)
        class_weight_map = {cls: float(max(counts) / cnt) for cls, cnt in zip(classes, counts)}
        sample_weights = np.array([class_weight_map[y] for y in y_t])

        model = XGBClassifier(**params, n_jobs=1, random_state=42)
        model.fit(X_t, y_t, sample_weight=sample_weights, verbose=False)

        y_pred = model.predict(X_v)
        scores.append(f1_score(y_v, y_pred, average='macro'))

    return float(np.mean(scores))

# ----------------------
# Run Optuna study
# ----------------------
sampler = TPESampler(seed=42)
study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    study_name='xgb_class_weighted_macroF1'
)

study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV macro-F1:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# ----------------------
# Train final model on full training set using class weights
# ----------------------
best = study.best_params.copy()
best.update({'eval_metric': 'mlogloss', 'random_state': 42})

classes, counts = np.unique(y_train_enc, return_counts=True)
final_class_weight_map = {cls: float(max(counts) / cnt) for cls, cnt in zip(classes, counts)}
final_sample_weights = np.array([final_class_weight_map[y] for y in y_train_enc])

final_xgb = XGBClassifier(**best, n_jobs=-1)
final_xgb.fit(X_train_arr, y_train_enc, sample_weight=final_sample_weights, verbose=False)

# Evaluate on untouched test set
y_test_pred_enc = final_xgb.predict(X_test_arr)
y_test_pred = le.inverse_transform(y_test_pred_enc)

print("\nFINAL Test Classification Report (class-weighted XGB):")
print(classification_report(y_test, y_test_pred, labels=le.classes_))
print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_test_pred, labels=le.classes_))

[I 2026-11-19 17:55:40,198] A new study created in memory with name: xgb_class_weighted_macroF1


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-11-19 17:55:40,739] Trial 0 finished with value: 0.9969967525439518 and parameters: {'n_estimators': 181, 'max_depth': 12, 'learning_rate': 0.06504856968981275, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.4936111842654619, 'gamma': 0.7799726016810132, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 0.9969967525439518.
[I 2026-11-19 17:55:43,636] Trial 1 finished with value: 0.9946937429291142 and parameters: {'n_estimators': 260, 'max_depth': 9, 'learning_rate': 0.001124579825911934, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.899465584480253, 'gamma': 1.0616955533913808, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 0.9969967525439518.
[I 2026-11-19 17:55:45,089] Trial 2 finished with value: 0.9954544316863482 and parameters: {'n_estimators': 156, 'max_depth': 7, 'learning_rate': 0.01174843954800703, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.76711

In [54]:
import joblib

joblib.dump(model,'model_xgb_new.pkl')

print("Model saved successfully as 'model_xgb_new.pkl'")

Model saved successfully as 'model_xgb_new.pkl'
